# Character training data generator
This notebook generates 42x42px PNG images of characters for training data.

In [6]:
import string
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from utils import preprocessing_remove_lines, preprocessing_split_by_characters

In [7]:
keys = list(string.ascii_lowercase) + [str(i) for i in range(10)]
char_dict = {k: [] for k in keys}
char_dir = 'chars_train_data/'
# Delete all files in the target folder
for filename in os.listdir(char_dir):
    file_path = os.path.join(char_dir, filename)
    if os.path.isfile(file_path) and filename.lower().endswith(".png"):
        os.remove(file_path)
        
os.makedirs(char_dir, exist_ok=True)

# Start character separation

In [8]:
wrong_count = 0
images_count = 0

# Loop through all png in train/
for (root,dirs,files) in os.walk('clean_train_data/',topdown=True):
    for file in tqdm(files, desc="Loading images"):
        if file.endswith('.png'):
            if file[1] == "_":
                os.remove(os.path.join(root, file))
                continue
            if file.split("-")[1].split(".png")[0] != "0":
                continue
                
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)
            processed_img = preprocessing_remove_lines(img) # Remove lines
            num_chars, char_images = preprocessing_split_by_characters(processed_img) # Split chars
            ground_truth = img_path.split("-")[0].split("/")[-1] # Get Captcha label

            # If wrong seperation count, do not add to chars_train_data
            images_count += 1
            if len(char_images) != len(ground_truth):
                wrong_count += 1
                continue

            # Correct seperated
            for i in range(len(ground_truth)):
                k = ground_truth[i]
                char_dict[k].append(char_images[i])

print(f"Total captcha count: {images_count}")
print(f"Wrong character sep count: {wrong_count}")
            

Loading images: 100%|██████████████████████████████████████████████████████████████| 8001/8001 [06:31<00:00, 20.44it/s]

Total captcha count: 7812
Wrong character sep count: 382


### Process character image after separation

In [9]:
def process_character_image(img):
    # --- Convert to grayscale ---
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # --- Invert so character = white (255), background = black (0) ---
    gray = 255 - gray
    
    # --- Threshold to make binary ---
    # _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # --- Find bounding box of character ---
    coords = cv2.findNonZero(gray)
    x, y, w, h = cv2.boundingRect(coords)
    
    # --- Crop + 3px padding ---
    pad = 3
    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + w + pad, gray.shape[1])
    y2 = min(y + h + pad, gray.shape[0])
    cropped = gray[y1:y2, x1:x2]

    # --- Normalize brightness: stretch intensity range to full 0–255 ---
    min_val, max_val = np.min(cropped), np.max(cropped)
    if max_val > min_val:  # avoid divide-by-zero if image is uniform
        cropped = (cropped - min_val) * (255.0 / (max_val - min_val))
        cropped = np.clip(cropped, 0, 255).astype(np.uint8)
    
    # Below: Pad or crop all letters to the same square (while avoiding stretching proportions)
    # --- Create black background ---
    target_size = 42
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    
    # --- If cropped image is larger than 32x32, center crop it ---
    h, w = cropped.shape
    if h > target_size or w > target_size:
        # Center crop
        start_y = max((h - target_size) // 2, 0)
        start_x = max((w - target_size) // 2, 0)
        end_y = start_y + target_size
        end_x = start_x + target_size
        cropped = cropped[start_y:end_y, start_x:end_x]
        h, w = cropped.shape
    
    # --- Compute top-left position to center the image ---
    y_offset = (target_size - h) // 2
    x_offset = (target_size - w) // 2
    
    # --- Paste cropped image onto canvas ---
    canvas[y_offset:y_offset + h, x_offset:x_offset + w] = cropped

    return canvas

### Save to PNG training data

In [10]:
# Save each image to PNG
for k, img_list in char_dict.items():
    for idx, img in tqdm(enumerate(img_list), desc=k):
        filename = f"{k}_{idx:05d}.png"  # e.g., a_0000.png
        path = os.path.join(char_dir, filename)
        processed_img = process_character_image(img)
        cv2.imwrite(path, processed_img)



a: 1196it [00:03, 353.99it/s]
b: 1259it [00:03, 351.27it/s]
c: 1216it [00:03, 358.50it/s]
d: 1271it [00:03, 337.03it/s]
e: 1268it [00:03, 348.95it/s]
f: 1238it [00:03, 312.04it/s]
g: 1263it [00:03, 342.65it/s]
h: 1242it [00:03, 356.59it/s]
i: 1236it [00:03, 365.63it/s]
j: 1180it [00:03, 360.89it/s]
k: 1238it [00:03, 368.83it/s]
l: 1232it [00:03, 367.30it/s]
m: 1245it [00:03, 363.34it/s]
n: 1292it [00:03, 363.47it/s]
o: 1219it [00:03, 354.26it/s]
p: 1266it [00:03, 355.69it/s]
q: 1287it [00:03, 356.95it/s]
r: 1236it [00:03, 362.33it/s]
s: 1219it [00:03, 359.08it/s]
t: 1234it [00:03, 364.47it/s]
u: 1202it [00:03, 361.92it/s]
v: 1240it [00:03, 358.64it/s]
w: 1208it [00:03, 356.76it/s]
x: 1272it [00:03, 335.78it/s]
y: 1187it [00:03, 342.65it/s]
z: 1229it [00:03, 347.63it/s]
0: 1240it [00:03, 346.33it/s]
1: 1246it [00:03, 339.47it/s]
2: 1172it [00:03, 339.10it/s]
3: 1252it [00:03, 341.49it/s]
4: 1229it [00:03, 338.16it/s]
5: 1187it [00:03, 316.25it/s]
6: 1223it [00:03, 317.17it/s]
7: 1174it 